 # Milestone 2: Data Quality dan Profiling

**Nama:** Afzal Raditya Dharma  
**NIM:** 202310370311236
 

In [1]:
from pathlib import Path
import duckdb
import polars as pl
from IPython.display import display

PROJECT_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                     if (p / 'data/raw/apbd_indonesia_2011_2026.parquet').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Unduh dataset APBD ke data/raw/ sebelum menjalankan cleaning.')
DATA_PATH = PROJECT_ROOT / 'data/raw/apbd_indonesia_2011_2026.parquet'
OUTPUT_PATH = PROJECT_ROOT / 'data/processed/apbd_indonesia_bersih.parquet'
df = pl.read_parquet(DATA_PATH)
print(f'Ukuran awal: {df.shape[0]:,} baris x {df.shape[1]} kolom')
df.head()

Ukuran awal: 3,132,863 baris x 11 kolom
Out[6]: 
shape: (5, 11)
┌───────┬───────┬─────────────┬──────────┬───┬────────────────┬───────────┬───────────┬────────────┐
│ tahun ┆ bulan ┆ id_provinsi ┆ provinsi ┆ … ┆ akun           ┆ anggaran  ┆ realisasi ┆ persentase │
│ ---   ┆ ---   ┆ ---         ┆ ---      ┆   ┆ ---            ┆ ---       ┆ ---       ┆ ---        │
│ i16   ┆ i8    ┆ str         ┆ str      ┆   ┆ str            ┆ f64       ┆ f64       ┆ f64        │
╞═══════╪═══════╪═════════════╪══════════╪═══╪════════════════╪═══════════╪═══════════╪════════════╡
│ 2011  ┆ 1     ┆ 01          ┆ Provinsi ┆ … ┆ Pendapatan     ┆ 7.0894e12 ┆ 7.6093e12 ┆ 107.33     │
│       ┆       ┆             ┆ Aceh     ┆   ┆                ┆           ┆           ┆            │
│ 2011  ┆ 1     ┆ 01          ┆ Provinsi ┆ … ┆ PAD            ┆ 7.9728e11 ┆ 8.0284e11 ┆ 100.7      │
│       ┆       ┆             ┆ Aceh     ┆   ┆                ┆           ┆           ┆            │
│ 2011  ┆ 1     ┆ 01       

# 1. Profiling dengan DuckDB SUMMARIZE

In [2]:
connection = duckdb.connect()
connection.register("dataset", df)

def hasil_polars(result):
    tabel = (result.to_arrow_table() if hasattr(result, "to_arrow_table")
             else result.fetch_arrow_table())
    return pl.from_arrow(tabel)

profil = hasil_polars(connection.execute("SUMMARIZE SELECT * FROM dataset"))
profil

Out[7]: 
shape: (11, 12)
┌────────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬─────────┬───────────┐
│ column_nam ┆ column_ty ┆ min       ┆ max       ┆ … ┆ q50       ┆ q75       ┆ count   ┆ null_perc │
│ e          ┆ pe        ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---     ┆ entage    │
│ ---        ┆ ---       ┆ str       ┆ str       ┆   ┆ str       ┆ str       ┆ i64     ┆ ---       │
│ str        ┆ str       ┆           ┆           ┆   ┆           ┆           ┆         ┆ decimal[9 │
│            ┆           ┆           ┆           ┆   ┆           ┆           ┆         ┆ ,2]       │
╞════════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═════════╪═══════════╡
│ tahun      ┆ SMALLINT  ┆ 2011      ┆ 2026      ┆ … ┆ 2019      ┆ 2022      ┆ 3132863 ┆ 0.00      │
│ bulan      ┆ TINYINT   ┆ 1         ┆ 12        ┆ … ┆ 6         ┆ 9         ┆ 3132863 ┆ 0.00      │
│ id_provins ┆ VARCHAR   ┆ 01        ┆ 38        ┆ … ┆ null      ┆

<ipython-input-7-4577bdfa8c07>:5: UserWarning: Arrow output on a DB-API result makes an extra copy of the rows, use con.sql(query).to_arrow_table() instead.
  tabel = (result.to_arrow_table() if hasattr(result, "to_arrow_table")


# 2. DETEKSI MISSING VALUES

In [3]:
jumlah_baris = len(df)
null_summary = (
    df.null_count()
    .transpose(include_header=True, header_name="kolom", column_names=["jumlah_null"])
    .with_columns(
        (pl.col("jumlah_null") / jumlah_baris * 100).round(2).alias("persen_null")
    )
    .filter(pl.col("jumlah_null") > 0)
    .sort("jumlah_null", descending=True)
)

if len(null_summary) == 0:
    print("Tidak ditemukan missing values (null) pada seluruh kolom dataset.")
else:
    display(null_summary)

Tidak ditemukan missing values (null) pada seluruh kolom dataset.


## 3. Rasio dan nilai yang tidak terdefinisi

Data sumber tidak memiliki null. Persentase sumber dipertahankan dalam
`persentase_sumber`, sedangkan `persentase` dihitung ulang dari nominal agar
konsisten. Untuk anggaran nol atau negatif, rasio penyerapan tidak ditafsirkan
sehingga diisi null, bukan 0%. Null turunan ini disengaja dan tidak berarti
kolom sumber hilang.

In [4]:
df_missing_handled = df.with_columns(
    pl.col('persentase').alias('persentase_sumber'),
    pl.when(pl.col('anggaran') > 0)
      .then(100 * pl.col('realisasi') / pl.col('anggaran'))
      .otherwise(None).alias('persentase'),
)
print(f'Null pada data sumber: {df.null_count().sum_horizontal().sum():,}')
print(f'Rasio tidak terdefinisi: {df_missing_handled["persentase"].null_count():,}')

Null pada data sumber: 0
Rasio tidak terdefinisi: 347,582


## 4. Penandaan anomali tanpa mengubah nominal

Nominal negatif tidak otomatis salah: pada data ini banyak terdapat di akun
pengeluaran pembiayaan, sehingga konvensi tanda perlu dikonfirmasi ke sumber.
Nominal `anggaran` dan `realisasi` dipertahankan tanpa clipping atau imputasi.
Rasio di luar 0–200% diberi flag untuk peninjauan, bukan dianggap melanggar aturan
hukum atau langsung dihapus. Rasio dapat melampaui 100% karena revisi anggaran,
perbedaan waktu pelaporan, atau kesalahan sumber; dataset ini belum cukup untuk
menetapkan penyebabnya. IQR global tidak dipakai karena skala pemda dan hierarki
akun berbeda. Pengaruh pencilan dianalisis melalui uji sensitivitas pada EDA.

In [5]:
df_outlier_handled = df_missing_handled.with_columns(
    (pl.col('anggaran') < 0).alias('flag_anggaran_negatif'),
    (pl.col('realisasi') < 0).alias('flag_realisasi_negatif'),
    (pl.col('anggaran') <= 0).alias('flag_rasio_tidak_terdefinisi'),
    ((pl.col('persentase') < 0) | (pl.col('persentase') > 200))
      .fill_null(False).alias('flag_rasio_ekstrem'),
)
display(df_outlier_handled.select(pl.col('^flag_.*$').sum()))
display(df_outlier_handled.filter(pl.col('flag_rasio_ekstrem'))
        .sort('persentase', descending=True)
        .select('tahun', 'bulan', 'pemda', 'akun', 'anggaran', 'realisasi', 'persentase').head(10))

flag_anggaran_negatif,flag_realisasi_negatif,flag_rasio_tidak_terdefinisi,flag_rasio_ekstrem
u32,u32,u32,u32
75086,75550,347582,180062


tahun,bulan,pemda,akun,anggaran,realisasi,persentase
i16,i8,str,str,f64,f64,f64
2024,12,"""Kab. Teluk Wondama""","""Pembiayaan Daerah""",360.0,1.1655e10,3.2376e9
2025,12,"""Kab. Sumenep""","""Penerimaan Kembali Pemberian P…",1000.0,8.6864127e7,8686412.7
2025,11,"""Kab. Sumenep""","""Penerimaan Kembali Pemberian P…",1000.0,8.0694201e7,8069420.1
2025,10,"""Kab. Sumenep""","""Penerimaan Kembali Pemberian P…",1000.0,7.9785111e7,7978511.1
2017,1,"""Kab. Kolaka Timur""","""Pengeluaran Pembiayaan Daerah""",197631.0,6.1106e9,3.0919e6
2017,1,"""Kab. Kolaka Timur""","""Penyertaan Modal Daerah""",197631.0,6.1106e9,3.0919e6
2017,2,"""Kab. Kolaka Timur""","""Pengeluaran Pembiayaan Daerah""",197631.0,6.1106e9,3.0919e6
2017,2,"""Kab. Kolaka Timur""","""Penyertaan Modal Daerah""",197631.0,6.1106e9,3.0919e6
2017,3,"""Kab. Kolaka Timur""","""Pengeluaran Pembiayaan Daerah""",197631.0,6.1106e9,3.0919e6


# 5. DEDUPLIKASI

In [6]:
KUNCI_BISNIS = ['tahun', 'bulan', 'id_provinsi', 'id_pemda', 'no_akun']
baris_unik = df_outlier_handled.select(KUNCI_BISNIS).n_unique()
unik_penuh = df_outlier_handled.unique(maintain_order=True)
konflik = unik_penuh.group_by(KUNCI_BISNIS).len().filter(pl.col('len') > 1)
if konflik.height:
    raise ValueError(f'Ada {konflik.height} kunci dengan nilai berbeda; tinjau sebelum deduplikasi.')
df_dedup = unik_penuh
print(f'Baris sebelum deduplikasi: {df_outlier_handled.height:,}')
print(f'Kunci unik: {baris_unik:,}')
print(f'Duplikat identik dihapus: {df_outlier_handled.height - df_dedup.height:,}')

Baris sebelum deduplikasi: 3,132,863
Kunci unik: 3,132,863
Duplikat identik dihapus: 0


# 6. VALIDASI DATA DASAR (DUCKDB)

In [7]:
connection.register('dataset_bersih', df_dedup)
validasi = hasil_polars(connection.execute("""
    SELECT COUNT(*) AS total_baris,
        SUM(CASE WHEN anggaran IS NULL OR realisasi IS NULL THEN 1 ELSE 0 END) AS pelanggaran_null_nominal,
        SUM(CASE WHEN bulan < 1 OR bulan > 12 THEN 1 ELSE 0 END) AS pelanggaran_bulan,
        SUM(CASE WHEN anggaran <= 0 AND persentase IS NOT NULL THEN 1
                 WHEN anggaran > 0 AND (persentase IS NULL OR
                 ABS(persentase - 100 * realisasi / anggaran) > 0.000001) THEN 1
                 ELSE 0 END) AS pelanggaran_rasio,
        SUM(CASE WHEN flag_anggaran_negatif <> (anggaran < 0)
                 OR flag_realisasi_negatif <> (realisasi < 0)
                 OR flag_rasio_tidak_terdefinisi <> (anggaran <= 0)
                 OR flag_rasio_ekstrem <> COALESCE(persentase < 0 OR persentase > 200, FALSE)
                 THEN 1 ELSE 0 END) AS pelanggaran_flag
    FROM dataset_bersih
"""))
for kolom in ['pelanggaran_null_nominal', 'pelanggaran_bulan', 'pelanggaran_rasio', 'pelanggaran_flag']:
    assert validasi[kolom][0] == 0, f'Validasi gagal: {kolom}'
assert df_dedup.select(KUNCI_BISNIS).n_unique() == df_dedup.height
expected = df.unique(maintain_order=True)
assert df_dedup.select(df.columns[:-1]).equals(expected.select(df.columns[:-1]))
assert df_dedup['persentase_sumber'].equals(expected['persentase'].rename('persentase_sumber'))
display(validasi)
connection.close()

<ipython-input-7-4577bdfa8c07>:5: UserWarning: Arrow output on a DB-API result makes an extra copy of the rows, use con.sql(query).to_arrow_table() instead.
  tabel = (result.to_arrow_table() if hasattr(result, "to_arrow_table")


total_baris,pelanggaran_null_nominal,pelanggaran_bulan,pelanggaran_rasio,pelanggaran_flag
i64,"decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]"
3132863,0,0,0,0


# 7. SIMPAN DATA BERSIH

In [8]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_dedup.write_parquet(OUTPUT_PATH)

print(f"Baris sebelum cleaning: {len(df):,}")
print(f"Baris sesudah cleaning: {len(df_dedup):,}")
print(f"Data bersih tersimpan di: {OUTPUT_PATH}")

Baris sebelum cleaning: 3,132,863
Baris sesudah cleaning: 3,132,863
Data bersih tersimpan di: D:\Kuliah\Koding\Semester_7\Analisis_Big_Data\tugas1-AfzalRaditya\data\processed\apbd_indonesia_bersih.parquet


## Catatan Milestone 2

- Seluruh 11 kolom sumber terisi; rasio turunan untuk anggaran nol/negatif sengaja null.
- Deduplikasi hanya menghapus baris identik. Kunci dengan nilai berbeda menghentikan
  pipeline agar laporan tidak dipilih secara arbitrer. Pada file saat ini ada
  3.132.863 kunci unik dan tidak ada duplikat.
- Nominal sumber dan persentase sumber dipertahankan. Empat flag menandai nominal
  negatif, rasio tidak terdefinisi, serta rasio ekstrem tanpa memotong nilainya.
- Validasi memeriksa null nominal, bulan, kesesuaian rasio/flag, keunikan kunci,
  serta kesamaan nominal dengan sumber. Nilai ekstrem tetap memerlukan verifikasi,
  sehingga lolos validasi struktur tidak berarti semua nominal telah terbukti benar.
- Hasil disimpan di `data/processed/apbd_indonesia_bersih.parquet` untuk EDA.